# GlowPick 전성분 검증 — 유성원
수업 4~7·9~11주차 방식 대응. 현재 OCR/Gemini 실험 결과는 없으며 이 노트북에서 직접 실행해 기록합니다. 정답은 웹 이미지 임시 전사, 실물 라벨 검증과 구분합니다. 무료 T4가 없으면 결제하지 말고 멈추세요.

## 1. 최종 ZIP 업로드와 안전하게 풀기
이 패키지 ZIP 하나를 선택하세요. 이전 ZIP이나 강의 ZIP이 아닙니다.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, json, sys
uploaded = files.upload()
zip_path = next(Path(name) for name in uploaded if name.endswith('.zip'))
destination = Path('/content/glowpick_input').resolve()
destination.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_path) as archive:
    for info in archive.infolist():
        target = (destination / info.filename).resolve()
        if not target.is_relative_to(destination) or (info.external_attr >> 16) & 0o170000 == 0o120000:
            raise ValueError('안전하지 않은 압축 경로')
    archive.extractall(destination)
root = next(p.parent for p in destination.rglob('README.md') if (p.parent/'data/pairs.json').exists())
sys.path.insert(0,str(root/'src'))
print('작업 폴더:',root)


## 2. 외부 API 없는 파일·비교 도구 점검
먼저 이것만 실행해도 됩니다. 단위시험 통과는 OCR 성능이 아닙니다.

In [ ]:
import subprocess, hashlib
subprocess.run([sys.executable,'-m','unittest','discover','-s',str(root/'tests'),'-p','test_*.py'],check=True)
images=json.loads((root/'data/images.json').read_text(encoding='utf-8'))
assert all(hashlib.sha256((root/x['path']).read_bytes()).hexdigest()==x['sha256'] for x in images)
pairs=json.loads((root/'data/pairs.json').read_text(encoding='utf-8'))
print('원본 이미지 해시 확인:',len(images),'정답 쌍:',len(pairs))


## 3. 원본 사진과 정답을 직접 보기
자동으로 맞는 정답이라고 간주하지 말고 원본과 대조하세요.

In [ ]:
from PIL import Image
from IPython.display import display
pair = pairs[0]
display(Image.open(root/pair['image_path']))
print(json.dumps(pair['ingredients'],ensure_ascii=False,indent=2))


## 4. Hugging Face 모델 실행 준비
수업의 processor→generate→decode 방식. T4 GPU 선택 후 설치합니다. Pillow는 업그레이드하지 않습니다. HF 다운로드·설치 실패 시 기록하고 멈추세요.

In [ ]:
import torch
assert torch.cuda.is_available(), '무료 T4 GPU를 선택하세요. 없으면 이 실험은 미실행으로 남깁니다.'
%pip -q install "transformers>=4.51,<5" "accelerate>=1,<2" "gradio>=5,<6"
import transformers
print('torch:',torch.__version__,'transformers:',transformers.__version__)


## 5. 실제 HF OCR와 정답 비교
정답을 모델에 보내지 않습니다. 한 번의 실행으로 5쌍을 평가하고 원응답·오류·모델 revision·버전을 저장합니다. 작은 글자/다단 구조 실패는 그대로 기록합니다. 이 셀을 다시 실행하기 전 이전 결과를 다운로드하세요.

In [ ]:
from hf_ocr import run_batch
runs=run_batch(root)
for run in runs:
    print(run['pair_id'],run['execution_status'],run['metrics'])


## 6. 원본 12장 예외 사례의 실제 시험(선택)
5쌍 평가와 별도입니다. 사용법·앞면 사진에 완전한 성분을 만들어 넣는지 확인합니다. 기대 상태는 평가용으로만 사용합니다.

In [ ]:
from hf_ocr import load_model, extract
processor, model = load_model()
cases=json.loads((root/'data/image_cases.json').read_text(encoding='utf-8'))
exception_runs=[]
for image,case in zip(images,cases):
    record=extract(Image.open(root/image['path']),processor,model)
    record['image_id']=image['image_id']
    record['expected_status']=case['expected_status']
    record['status_match']=(record['response']['status']==case['expected_status']) if record['response'] else False
    exception_runs.append(record)
    (root/'results/hf_exception_runs.json').write_text(json.dumps(exception_runs,ensure_ascii=False,indent=2),encoding='utf-8')
print('전체 시도:',len(exception_runs),'상태 일치:',sum(x['status_match'] for x in exception_runs))


## 7. Gradio 검증 화면
HF 결과의 response 객체를 복사해 붙여 넣어 비교합니다. 이 화면 자체는 OCR을 호출하지 않습니다.

In [ ]:
import runpy
app=runpy.run_path(str(root/'space/app.py'))
app['demo'].launch(share=True)


## 8. Gemini 설명 인계 시험(선택)
9~10주차 SDK와 Interactions 형태입니다. Colab 비밀값 GEMINI_API_KEY, GEMINI_MODEL을 설정하세요. 본인 무료 계정에서 모델 가용성·한도를 먼저 확인합니다. 이 셀은 기본값으로 호출을 막습니다.

In [ ]:
# 이 선택 설치가 실패해도 앞선 OCR 결과를 보존하세요.
%pip -q install google-genai==2.12.1


In [ ]:
from google.colab import userdata
from gemini_handoff import analyze
# 본인이 원문 전체를 대조한 후에만 True로 변경
reviewed=False
chosen=next((x for x in runs if x['execution_status']=='success' and x['response']['status']=='complete'),None)
if chosen is None or not reviewed:
    print('실제 complete 응답과 사람 확인이 필요합니다. API 호출하지 않음.')
else:
    result=analyze(chosen['response'],['건조'],api_key=userdata.get('GEMINI_API_KEY'),
                   model_name=userdata.get('GEMINI_MODEL'),reviewed=reviewed)
    (root/'results/gemini_run.json').write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding='utf-8')
    print(result)


## 9. 실제 실행 결과 내려받기
성공·실패 원응답과 환경 버전을 함께 보관합니다. 키는 결과에 저장하지 않습니다.

In [ ]:
import shutil, datetime
stamp=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
result_zip=shutil.make_archive('/content/GlowPick_results_'+stamp,'zip',root,'results')
files.download(result_zip)
